## TF-IDF Based Content Recommender

In this task, we implement a content-based recommendation system using TF-IDF (Term Frequency–Inverse Document Frequency). 

Each movie is represented using its genres, which are treated as textual features. TF-IDF helps convert these textual features into numerical vectors by assigning importance to each term.

Movies with similar genre compositions will have similar TF-IDF vectors.

In [25]:
import pandas as pd
import numpy as np

In [26]:
movies = pd.read_csv("dataset/movies.csv")
ratings = pd.read_csv("dataset/ratings.csv")

In [27]:
movies['genres'] = movies['genres'].str.replace('|', ' ')
movies = movies[['movieId','title','genres']]

In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(movies['genres'])

print(tfidf_matrix.shape)

(9742, 23)


In [29]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [30]:
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

In [31]:
def recommend_movies(title, top_n=5):
    idx = indices[title]
    
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    sim_scores = sim_scores[1:top_n+1]
    
    movie_indices = [i[0] for i in sim_scores]
    
    return movies.iloc[movie_indices][['title']], sim_scores

In [32]:
recommend_movies("Toy Story (1995)")

(                                               title
 1706                                     Antz (1998)
 2355                              Toy Story 2 (1999)
 2809  Adventures of Rocky and Bullwinkle, The (2000)
 3000                Emperor's New Groove, The (2000)
 3568                           Monsters, Inc. (2001),
 [(1706, np.float64(1.0)),
  (2355, np.float64(1.0)),
  (2809, np.float64(1.0)),
  (3000, np.float64(1.0)),
  (3568, np.float64(1.0))])

## User Profile Based Content Recommender

In this task, we personalize recommendations by building a user profile.

Each user profile is constructed as a weighted combination of the TF-IDF vectors of movies they have rated. The weights are based on the ratings given by the user.

This approach ensures that movies rated higher contribute more to the user’s preference profile.

### Mathematical Formulation

The user profile vector is computed as:

Pu = Σ (ru,m × fm) / Σ (ru,m)

Where:
- Pu = User profile vector
- ru,m = Rating given by user u to movie m
- fm = Feature vector of movie m

In [21]:
movie_data = pd.merge(ratings, movies, on='movieId')
movie_data.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure Animation Children Comedy Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy Romance
2,1,6,4.0,964982224,Heat (1995),Action Crime Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime Mystery Thriller


## Constructing User Profile

We construct a user profile by:
1. Selecting movies rated by the user
2. Extracting their TF-IDF vectors
3. Weighting them using user ratings
4. Computing a normalized weighted average

In [39]:
def build_user_profile(user_id):
    user_movies = movie_data[movie_data['userId'] == user_id]
    
    movie_indices = user_movies['movieId'].apply(
        lambda x: movies[movies['movieId'] == x].index[0]
    )
    
    user_tfidf = tfidf_matrix[movie_indices]
    
    ratings_weights = user_movies['rating'].values.reshape(-1,1)
    
    user_profile = np.sum(user_tfidf.multiply(ratings_weights), axis=0) / np.sum(ratings_weights)
    
    return np.asarray(user_profile)

## Generating Recommendations for a User

Once the user profile is created, we compute similarity between the user profile and all movie vectors using cosine similarity.

Movies are then ranked based on similarity scores.

In [40]:
def recommend_for_user(user_id, top_n=5):
    user_profile = build_user_profile(user_id)
    
    similarity = cosine_similarity(user_profile.reshape(1, -1), tfidf_matrix)
    
    sim_scores = list(enumerate(similarity[0]))
    
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    movie_indices = [i[0] for i in sim_scores[:top_n]]
    
    return movies.iloc[movie_indices][['title']]

In [41]:
recommend_for_user(1)

,title
8597,Dragonheart 2: A New Beginning (2000)
6570,"Hunting Party, The (2007)"
4005,Flashback (1990)
4681,The Great Train Robbery (1978)
4409,Charlie's Angels: Full Throttle (2003)
